# E1 — Clean Baseline (no poisoning) on BERT-base / SST-2
No trigger, no label flipping. This is the reference point every other experiment (E2, E3...) is compared against.

In [1]:
!pip install transformers datasets scikit-learn --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [3]:
ds = load_dataset("stanfordnlp/sst2")
print(ds)

clean_train_df = pd.DataFrame({"sentence": ds["train"]["sentence"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["validation"]["sentence"], "label": ds["validation"]["label"]})
print(clean_train_df.shape, clean_valid_df.shape)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
(67349, 2) (872, 2)


## Tokenization -- no poisoning step here, straight to tokenizing raw clean data

In [4]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train_ds = to_hf_dataset(clean_train_df)
valid_ds = to_hf_dataset(clean_valid_df)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

## Train

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

args = TrainingArguments(
    output_dir="./results_e1_clean",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                   compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.173271,0.253862,0.922018,0.935185,0.909910,0.922374
2,0.117054,0.294620,0.922018,0.903433,0.948198,0.925275
3,0.083630,0.324869,0.932339,0.924945,0.943694,0.934225


TrainOutput(global_step=12630, training_loss=0.13726251583106916, metrics={'train_runtime': 1815.1137, 'train_samples_per_second': 111.314, 'train_steps_per_second': 6.958, 'total_flos': 6645099925290240.0, 'train_loss': 0.13726251583106916, 'epoch': 3.0})

## Evaluate -- CACC only (no trigger exists yet, so no ASR here)

In [6]:
preds = np.argmax(trainer.predict(valid_ds).predictions, axis=-1)
cacc = accuracy_score(clean_valid_df["label"], preds)
p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], preds, average="binary")
cm = confusion_matrix(clean_valid_df["label"], preds)

e1_results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
print(e1_results)
print("Confusion matrix:\n", cm)
print(classification_report(clean_valid_df["label"], preds))

{'CACC': 0.9323394495412844, 'Precision': 0.9249448123620309, 'Recall': 0.9436936936936937, 'F1': 0.9342251950947603}
Confusion matrix:
 [[394  34]
 [ 25 419]]
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       428
           1       0.92      0.94      0.93       444

    accuracy                           0.93       872
   macro avg       0.93      0.93      0.93       872
weighted avg       0.93      0.93      0.93       872



## Save
This exact model is reused later as: (a) the E1 baseline row in your results table, and (b) the **surrogate model** for E3/CBS confidence scoring.

In [7]:
model.save_pretrained("./models/e1_clean")
tokenizer.save_pretrained("./models/e1_clean")
print("saved e1_clean")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e1_clean


## Summary

In [8]:
summary_df = pd.DataFrame({"clean_baseline": e1_results}).T
summary_df

,CACC,Precision,Recall,F1
clean_baseline,0.932339,0.924945,0.943694,0.934225


In [ ]:
summary_df.to_excel("E1_results_summary.xlsx", index=True)
print("Saved: E2_resukts_summary.xlsx")

Saved: E2_resukts_summary.xlsx


: 